In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from scipy.stats import ttest_ind, ttest_1samp

import sys
sys.path.append('./src')
from fid import calculate_fid
from data import PPCI

# Experriment settings 
(6480 total models)

- **Encoder**: clip, clip_large, dino, mae, vit, vit_large
- **Token**: class, mean, all
- **Splitting criteria**: experiment, experiment_easy, position, position_easy, random, random_easy
- **Task**: all, or
- **Hidden Layers**: 1, 2
- **Learning Rate**: 0.05, 0.005, 0.0005
- **Seed** = 0, 1, 2, 3, 4

In [ ]:
results = pd.read_csv("~/ISTAnt/results/istant_lq/experiments_result.csv", index_col=0)
mask = (results["task"]=="or") & (results["split_criteria"]=="experiment_easy")
results[mask].sort_values(by='acc', ascending=False)

In [ ]:
results = pd.read_csv("~/ISTAnt/results/istant_hq/experiments_result.csv", index_col=0)
results["EAD_"] = results["TEB"]+results["EAD"]

results[(results["task"]=='or') & (results["bacc"]>0.95)].sort_values(by='EAD_', ascending=True)

## Annotation Sampling Mechanism

In [ ]:
quality = "hq"
results = pd.read_csv(f"~/ISTAnt/results/istant_{quality}/experiments_result.csv", index_col=0)

task = results.task == 'or'
color = results.color == 'or'
results = results[task & color].sort_values("bacc", ascending=False)
n = 200

results["TERB"] = results["TEB"] / results["EAD"]
random_easy = results[results["split_criteria"] == "random_easy"][:n]
experiment_easy = results[results["split_criteria"] == "experiment_easy"][:n]
position_easy = results[results["split_criteria"] == "position_easy"][:n]

random = results[results["split_criteria"] == "random"][:n]
experiment = results[results["split_criteria"] == "experiment"][:n]
position = results[results["split_criteria"] == "position"][:n]

k = 2.2
fig, axs = plt.subplots(1, 2, figsize=(8, 4))

# plot violin plot
axs[0].violinplot([random_easy["TERB"], experiment_easy["TERB"], position_easy["TERB"]],
                  showmeans=False,
                  showmedians=True)
axs[0].axhline(y=0, color='black', linestyle='--', linewidth=1)
axs[0].set_ylim(-k, k)
axs[0].set_title(r"Many annotations ($|\mathcal{D}_s| \gg |\mathcal{D}_u|$)")

for i, y in zip([1,2,3], [random_easy["TERB"], experiment_easy["TERB"], position_easy["TERB"]]):
    x = np.random.normal(i, 0.04, size=len(y))
    axs[0].scatter(x, y, alpha=1, color='cornflowerblue', s=10, marker='*', linewidths=0)

# plot box plot
axs[1].violinplot([random["TERB"], experiment["TERB"], position["TERB"]],
                  showmeans=False,
                  showmedians=True)
axs[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axs[1].set_title(r"Few annotations ($|\mathcal{D}_s| \ll |\mathcal{D}_u|$)")
axs[1].set_ylim(-k, k)

for i, y in zip([1,2,3], [random["TERB"], experiment["TERB"], position["TERB"]]):
    x = np.random.normal(i, 0.04, size=len(y))
    axs[1].scatter(x, y, alpha=1, color='cornflowerblue', s=10, marker='*', linewidths=0)

# adding horizontal grid lines
for ax in axs:
    ax.yaxis.grid(True)
    ax.set_xticks([y + 1 for y in range(3)],
                  labels=['Random', 'Experiment', 'Position'])
axs[0].set_ylabel('TERB')

# save plot as pdf
plt.tight_layout()
plt.savefig(f"results/istant_{quality}/sampling_n{n}.pdf")
plt.show()

In [ ]:
# t test random_easy has mean 0
t, pvalue = ttest_1samp(random_easy["TEB"], 0, alternative='two-sided')
print(f"Random Easy: t={t:.4f}, p={pvalue}, mu: {random_easy['TEB'].mean()*100:.2f}%")
t, pvalue = ttest_1samp(random["TEB"], 0, alternative='two-sided')
print(f"Random: t={t:.4f}, p={pvalue}, mu: {random['TEB'].mean()*100:.2f}%")
t, pvalue = ttest_1samp(experiment_easy["TEB"], 0, alternative='two-sided')
print(f"Experiment Easy: t={t:.4f}, p={pvalue}, mu: {experiment_easy['TEB'].mean()*100:.2f}%")
t, pvalue = ttest_1samp(experiment["TEB"], 0, alternative='two-sided')
print(f"Experiment: t={t:.4f}, p={pvalue}, mu: {experiment['TEB'].mean()*100:.2f}%")
t, pvalue = ttest_1samp(position_easy["TEB"], 0, alternative='two-sided')
print(f"Position Easy: t={t:.4f}, p={pvalue}, mu: {position_easy['TEB'].mean()*100:.2f}%")
t, pvalue = ttest_1samp(position["TEB"], 0, alternative='two-sided')
print(f"Position: t={t:.4f}, p={pvalue}, mu: {position['TEB'].mean()*100:.2f}%")

## Misleading Objective

In [ ]:
quality = "hq"
results = pd.read_csv(f"~/ISTAnt/results/istant_{quality}/experiments_result.csv", index_col=0)
n = 10000
task_name = "or"
color_name = "or"

split = (results["split_criteria"] == "random_easy") | (results["split_criteria"] == "experiment_easy") | (results["split_criteria"] == "position_easy")
task = results.task == task_name
color = results.color == color_name
results = results[~split & task & color].sort_values("bacc", ascending=False)[:n]
print(f"lowest Balanced Accuracy: {results['bacc'].iloc[-1]}")

results["TEB_val"] = results["TEB_val"].abs()
results["TEB"] = results["TEB"].abs()
results["TEB_bin"] = results["TEB"].abs()

corr_matrix = results[["loss_val", "acc_val", "bacc_val", "TEB_val", "acc", "bacc", "TEB"]].corr(method='spearman')
corr_matrix.columns = ['BCE Loss', 'Accuracy', 'Bal. Accuracy', '|TEB|', 'Accuracy', 'Bal. Accuracy', '|TEB|']

fig, ax = plt.subplots()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', color='black')
cax = ax.matshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.axvline(x=3.5, color='black', linewidth=1, linestyle='--')
plt.axhline(y=3.5, color='black', linewidth=1, linestyle='--')
# add footnote at the bottom right
#plt.text(9, 6.4, "$_{val}$: Validation set\n$_{\mathcal{D}}$: Full dataset ($\mathcal{D}=\mathcal{D}_s \cup \mathcal{D}_u$)", fontsize=9, color='black')
colorbar = fig.colorbar(cax)
colorbar.set_label('Spearman rank-order correlation')
# save pdf
plt.savefig(f"results/istant_{quality}/evaluation_{color_name}_nolegend.pdf", bbox_inches='tight')
plt.show()

In [ ]:
quality = "hq"
results = pd.read_csv(f"~/ISTAnt/results/istant_{quality}/experiments_result.csv", index_col=0)
n = 10000
task_name = "or"
color_name = "or"

split = (results["split_criteria"] == "random_easy") | (results["split_criteria"] == "experiment_easy") | (results["split_criteria"] == "position_easy")
task = results.task == task_name
color = results.color == color_name
results = results[~split & task & color].sort_values("bacc", ascending=False)[:n]
print(f"lowest Balanced Accuracy: {results['bacc'].iloc[-1]}")

print(len(results))
results["TEB_val"] = results["TEB_val"].abs()
results["TEB"] = results["TEB"].abs()
results["TEB_bin"] = results["TEB"].abs()

corr_matrix = results[["loss_val", "acc_val", "bacc_val", "TEB_val", "acc", "bacc", "TEB"]].corr(method='spearman')
corr_matrix.columns = ['BCE Loss$_{val}$', 'Accuracy$_{val}$', 'Bal. Accuracy$_{val}$', '|TEB|$_{val}$', 'Accuracy$_{\mathcal{D}}$', 'Bal. Accuracy$_{\mathcal{D}}$', '|TEB|$_{\mathcal{D}}$']

fig, ax = plt.subplots()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', color='black')
cax = ax.matshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.axvline(x=3.5, color='black', linewidth=1, linestyle='--')
plt.axhline(y=3.5, color='black', linewidth=1, linestyle='--')
# add footnote at the bottom right
#plt.text(9, 6.4, "$_{val}$: Validation set\n$_{\mathcal{D}}$: Full dataset ($\mathcal{D}=\mathcal{D}_s \cup \mathcal{D}_u$)", fontsize=9, color='black')
colorbar = fig.colorbar(cax)
colorbar.set_label('Spearman rank-order correlation')
# save pdf
plt.savefig(f"results/istant_{quality}/evaluation_{color_name}.pdf", bbox_inches='tight')
plt.show()

## Representation Bias

In [ ]:
quality = "hq"
np.random.seed(4)
results = pd.read_csv(f"~/ISTAnt/results/istant_{quality}/experiments_result.csv", index_col=0)
n = 20

split = (results["split_criteria"] == "random_easy") | (results["split_criteria"] == "experiment_easy") | (results["split_criteria"] == "position_easy")
task = results.task == 'or'
color = results.color == 'or'
results = results[task & color].sort_values("bacc", ascending=False)

# rename encoders
results["encoder"] = results["encoder"].replace({
    "vit_large": "ViT-L",
    "vit": "ViT-S",
    "clip_large": "CLIP-ViT-L",
    "clip": "CLIP-ViT-S",
    "mae": "MAE",
    "dino": "DINOv2"})


results["TERB"] = results["TEB"] / results["EAD"]
results = results.groupby("encoder").head(n)


encoders = results['encoder'].unique()
encoders_colors = {encoder: np.random.rand(3,) for encoder in encoders}
results['encoder_color'] = results['encoder'].map(encoders_colors)
plt.figure(figsize=(6, 4))
for encoder, color in encoders_colors.items():
    plt.scatter(results.loc[results['encoder'] == encoder, 'bacc'], 
                results.loc[results['encoder'] == encoder, 'TERB'],
                color=color, label=encoder, alpha=0.8)
# add hline in 0
k = 0.65
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
# add grid
plt.grid(True)
plt.xlabel("Balanced Accuracy")
plt.ylabel("TERB")
# set ylim
plt.ylim(-k, k)
handles, labels = plt.gca().get_legend_handles_labels()
order = np.argsort(labels)
plt.legend(np.array(handles)[order], np.array(labels)[order])
plt.tight_layout()
plt.savefig(f"results/istant_{quality}/encoders_n{n}.pdf")
plt.show()


results = results.groupby("encoder").head(n).groupby("encoder").mean()[["acc", "bacc", "TEB", "TEB_bin"]].sort_values("bacc", ascending=False)
results


In [ ]:
encoders = ["dino", "clip_large", "clip", "mae", "vit", "vit_large"]
splits = ["experiment", "experiment_easy", "position", "position_easy", "random", "random_easy"]
tokens = ["class", "mean"]
# save in pandas dataframe
results = pd.DataFrame(columns=["encoder", "token", "split", "fid"])
for encoder in encoders:
    for token in tokens:
        for split in splits:
            dataset = PPCI(encoder = encoder,
                        token = token,
                        #task = "or",
                        split_criteria = split,
                        environment = "supervised",
                        #batch_size = 256,
                        #num_proc = 4,
                        #verbose = True,
                        data_dir = 'data/istant_hq',
                        results_dir = 'results/istant_hq')
            embeddings = np.array(dataset.supervised["X"])
            # normalize embeddings
            embeddings = (embeddings - embeddings.mean(axis=0)) / embeddings.std(axis=0)
            A = embeddings[dataset.supervised["split"]]
            B = embeddings[~dataset.supervised["split"]]
            fid = calculate_fid(A, B)
            print(f"FID ({encoder}, {token}, {split},): {fid}")
            results = results.append({"encoder": encoder, "token": token,  "split": split, "fid": fid}, ignore_index=True)
results.to_csv("results/istant_hq/fid.csv")

In [ ]:
# load results
results = pd.read_csv("results/istant_hq/fid.csv", index_col=0)
split_name = "position"
results[(results["split"]==split_name) | (results["split"]==f"{split_name}_easy")].groupby("encoder").agg(["mean", "std"])["fid"]

## Discretization Bias

In [ ]:
quality = "hq"
results = pd.read_csv(f"~/ISTAnt/results/istant_{quality}/experiments_result.csv", index_col=0)
n = 100000

split = (results["split_criteria"] == "random_easy") | (results["split_criteria"] == "experiment_easy") | (results["split_criteria"] == "position_easy")
task = results.task == 'or'
color = results.color == 'or'
#results = results[~split & task & color].sort_values("bacc", ascending=False)[:n]
print(len(results))
print(f"lowest Balanced Accuracy: {results['bacc'].iloc[-1]}")


t_statistic, p_value = ttest_ind(results['TEB'], results['TEB_bin'], alternative='two-sided')
print(f"TEB=TEB_bin: p-value={p_value}, t-statistic={t_statistic}")
t_statistic, p_value = ttest_ind(abs(results['TEB']), abs(results['TEB_bin']), alternative='less')
print(f"|TEB|=|TEB_bin|: p-value={p_value}, t-statistic={t_statistic}")
t_statistic, p_value = ttest_1samp(results['TEB'], 0, alternative='two-sided')
print(f"TEB=0: p-value={p_value}, t-statistic={t_statistic}")
t_statistic, p_value = ttest_1samp(results['TEB_bin'], 0, alternative='two-sided')
print(f"TEB_bin=0: p-value={p_value}, t-statistic={t_statistic}")

k = 2
# boxplots TEB and TEB_b
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
#fig.suptitle(f"TEB and TEB_b (ISTAnt {quality})")
axs[0].boxplot(abs(results['TEB']/results['EAD']), labels=["TERB"])
axs[0].set_title("TERB")
axs[0].axhline(y=0, color='black', linestyle='--')
axs[0].set_ylim(-k, k)
axs[1].boxplot(abs(results['TEB_bin']/results['EAD']), labels=["TERB_b"])
axs[1].set_title("TERB_bin")
axs[1].set_ylim(-k, k)
axs[1].axhline(y=0, color='black', linestyle='--')
plt.show()